# ContinuityGuard · RASCALS
## 01 — Native LinuxONE training and evaluation

All telemetry below is generated by our controlled service simulator. No hospital or patient data is used. Models learn only observable current/past features. Train, calibration, validation and test contain independent runs and distinct seeds.

Run all cells on the supplied LinuxONE Jupyter environment. This notebook calls the repository's original, inspectable implementation.

In [ ]:
from pathlib import Path
import os, sys, platform, json
ROOT = Path.cwd()
if not (ROOT / 'continuityguard').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'continuityguard').is_dir(), 'Open this notebook from the project notebooks folder'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print('Team: RASCALS')
print('Architecture:', platform.machine())
print('Python:', platform.python_version())
print('Project:', ROOT)
assert platform.machine() == 's390x', 'This submission notebook is intended to run on LinuxONE'


### Train, calibrate, select, then evaluate
Three forecast candidates and their alert policies compete jointly on validation incident precision/early recall under a false-alarm budget. A separate calibration split fits sigmoid calibration; PR-AUC is also reported. Final testing uses fresh 70000-series seeds after v1 and v2 development reports were inspected; incident matching is aligned to the 60-second horizon. Isolation Forest learns normal behavior including healthy demand changes.

In [ ]:
from continuityguard.train import main
main(ROOT / 'artifacts')

### Incident-level results
Misses and late detections are included. No point adjustment is used. Lead time is reported among early detections alongside the overall early-warning recall.

In [ ]:
import pandas as pd
report = json.loads((ROOT / 'artifacts/report.json').read_text())
columns = ['incidents', 'incident_precision', 'incident_recall', 'early_warning_recall', 'median_early_lead_seconds', 'false_alarms_per_healthy_service_hour', 'missed']
display(pd.DataFrame(report['systems']).T[columns])
print('Chosen forecast:', report['selected_model'])
print('Alert policy:', report['policy'])
print('Forecast test:', {k: v for k,v in report['forecast_test'].items() if k != 'reliability_bins'})


In [ ]:
from IPython.display import display, Image
display(Image(filename=str(ROOT / 'artifacts/benchmark.png')))
display(Image(filename=str(ROOT / 'artifacts/example-timeline.png')))

### Stress test and paired response
Abrupt failures are unseen during fitting and may provide no warning precursor. Response comparisons share the same exogenous demand and random seed. Their effect sizes depend on the declared simulator assumptions.

In [ ]:
print('Splits:', report['split'])
print('Unseen abrupt failures:', {k:v for k,v in report['unseen_abrupt_failures'].items() if k != 'incident_details'})
display(pd.DataFrame(report['paired_responses']))
print('Model integrity:', report['manifest']['sha256'])
print('Scope:', report['scope'])
print('Limitations:', *report['limitations'], sep='\n- ')
